# Write Data Quality Rules for Bronze → Silver — Solution


## Data Quality Rules

| # | Rule Name | Checks | Action on Failure |
|---|-----------|--------|-------------------|
| 1 | Primary key not null | `ticket_id` must be non-null | **Reject** — send to dead-letter table; alert pipeline owner |
| 2 | Primary key uniqueness | No duplicate `ticket_id` values | **Deduplicate** — keep record with latest `created_at` timestamp |
| 3 | Timestamp validity | `created_at` must parse as ISO 8601 datetime | **Reject** — unparseable dates cannot be placed in time partitions |
| 4 | Email format | `customer_email` must match RFC 5322 pattern (`*@*.*`) | **Flag** — set `email_valid = false`; include record but mark for review |
| 5 | Required field completeness | `subject` must be non-empty string | **Default** — set to `"(no subject)"` if empty |
| 6 | Numeric range validation | `resolution_time_hrs` must be null OR >= 0 | **Reject** — negative values indicate data corruption |

## Record-by-Record Analysis

**Record 1** (TKT-90421, status=open):
- ✅ All rules pass
- Result: **Included in silver**

**Record 2** (TKT-90421, status=resolved):
- ⚠️ Rule 2 (uniqueness): Duplicate `ticket_id` with Record 1
- Action: Keep Record 2 (later status = more complete). Record 1 discarded.
- Result: **Included in silver** (wins dedup)

**Record 3** (ticket_id=null):
- ❌ Rule 1 (PK not null): `ticket_id` is null → **Rejected**
- Also fails: Rule 4 (invalid email), Rule 5 (empty subject), Rule 6 (negative hours)
- Result: **Rejected to dead-letter**

**Record 4** (TKT-90499, invalid date):
- ❌ Rule 3 (timestamp validity): `"invalid-date"` cannot parse → **Rejected**
- Result: **Rejected to dead-letter**

## Silver Table Schema

```sql
CREATE TABLE silver.support_tickets (
    ticket_id       STRING       NOT NULL,  -- PK
    created_at      TIMESTAMP    NOT NULL,  -- UTC
    customer_email  STRING,                 -- nullable
    email_valid     BOOLEAN      NOT NULL,  -- quality flag
    subject         STRING       NOT NULL,  -- defaulted if empty
    priority        STRING       NOT NULL,  -- enum: low, high, critical
    status          STRING       NOT NULL,  -- enum: open, resolved, escalated
    assigned_agent  STRING,                 -- nullable (unassigned tickets exist)
    resolution_time_hrs  DECIMAL(10,2),     -- nullable until resolved
    _ingested_at    TIMESTAMP    NOT NULL,  -- pipeline metadata
    _source_file    STRING       NOT NULL   -- lineage
)
PARTITIONED BY (DATE(created_at))
```

## Deduplication Strategy

- **Key:** `ticket_id`
- **Winner:** Record with the latest `created_at` timestamp (same ID = status update from source system)
- **Method:** Iceberg MERGE INTO on `ticket_id`; when matched and source `created_at` > target `created_at`, update all fields
- **Timing:** Applied during bronze→silver ETL (not at ingestion)